In [5]:
%pip install langchain
%pip install langchain_openai
%pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00


In [6]:
import os
from langchain.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults


In [7]:
base_url = "https://inference-on-ibm.edgecollaborate.com/Llama-3.1-405B-Instruct/v1"
base_key = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJNWjF6aVpTaHRIVW1ELWoway02R2pIdkJQVUpyQkg2dWt5Wi1ial9RSDVnIn0.eyJleHAiOjE3NTY4OTczMjEsImlhdCI6MTc0OTEyMTMyMSwianRpIjoiNjQ2ZGY1MGMtZmUwOS00NDBhLTk3MzMtZDVlYjVlMTg4M2UyIiwiaXNzIjoiaHR0cHM6Ly9pbmZlcmVuY2Utb24taWJtLmVkZ2Vjb2xsYWJvcmF0ZS5jb20vcmVhbG1zL21hc3RlciIsImF1ZCI6ImFjY291bnQiLCJzdWIiOiJlM2QxNDA2ZC0yNGMwLTRlZjMtOWIxMi05Zjg3Y2Q4OGFhMjciLCJ0eXAiOiJCZWFyZXIiLCJhenAiOiJpYm0tdXNlciIsImFjciI6IjEiLCJyZWFsbV9hY2Nlc3MiOnsicm9sZXMiOlsiZGVmYXVsdC1yb2xlcy1tYXN0ZXIiLCJvZmZsaW5lX2FjY2VzcyIsInVtYV9hdXRob3JpemF0aW9uIl19LCJyZXNvdXJjZV9hY2Nlc3MiOnsiYWNjb3VudCI6eyJyb2xlcyI6WyJtYW5hZ2UtYWNjb3VudCIsIm1hbmFnZS1hY2NvdW50LWxpbmtzIiwidmlldy1wcm9maWxlIl19fSwic2NvcGUiOiJlbWFpbCBwcm9maWxlIiwiY2xpZW50SG9zdCI6IjE2OS42My4xODkuNjgiLCJlbWFpbF92ZXJpZmllZCI6ZmFsc2UsInByZWZlcnJlZF91c2VybmFtZSI6InNlcnZpY2UtYWNjb3VudC1pYm0tdXNlciIsImNsaWVudEFkZHJlc3MiOiIxNjkuNjMuMTg5LjY4IiwiY2xpZW50X2lkIjoiaWJtLXVzZXIifQ.T29ja1v_eYM_298rb01TO3knpbO9JWv3RdLNB0nPgHmS1rQLxp3u0ct2qoBIqhKVSijvkCK1AimZbFTHWk5xH5VNxbwZIr7176QCHySQExo-YzM_HsWMcdTXEMiET8QchU8PAoD0NRwAQvwKD-q4SMEGvZqjEFxhSBBFkkhQ4mRnex645iNxirZ8bpv3taXB1JJApaW5Xio9Ik1-rzqtYw7C1DOgqsveMYikGA__PjrmbxMlpVTwSKHTz6kEhBqZdCA1HrK45NUO782Cdlg-fiBwwRRyB4A8xKG6RGe7Hy0qlaN-ogj273lP6vkbHvtP1cywYJsS0bo40dHeBT7PEg"
llm = ChatOpenAI(model="meta-llama/Llama-3.1-405B-Instruct",openai_api_key=base_key,openai_api_base=base_url)

In [8]:
os.environ["TAVILY_API_KEY"]="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"

In [9]:
system_prompt = """
Answer the following questions as best you can. You have access to the
following tools:

{tools}

Use the following format:

Question: the input question you must answer

Thought: you should always think about what to do

Action: the action to take, should be one of [{tool_names}]

Action Input: the input to the action

Observation: the result of the action

... (this Thought/Action/Action Input/Observation can repeat N times)

Thought: I now know the final answer

Final Answer: the final answer to the original input question. Give answer in max 100 words

Begin!

Question: {input}

Thought:{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(system_prompt)


In [12]:
from langchain_core.tools import tool
import requests

from datetime import datetime
@tool
def get_location_from_ip():
    """Get the geographical location based on the IP address."""
    try:
        response = requests.get("https://ipinfo.io/json")
        data = response.json()
        print("got data")
        if 'loc' in data:
            data = f"Country: {data.get('country', 'N/A')}"
            return data
        else:
            return "Country could not be determined"
    except Exception as e:
        return f"Error occurred: {e}"

@tool
def get_current_datetime() -> str:
    """Get the current date and time"""
    return datetime.now().strftime("%Y-%m %H:%M:%S")

In [13]:
tools = [get_current_datetime,get_location_from_ip,TavilySearchResults(max_results=1)]

question = """Give me the cheapest flight from my current location to Taiwan in the next 24 hrs"""

agent = create_react_agent(llm,tools,prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input": question})



> Entering new AgentExecutor chain...
I need to find the user's current location to determine the origin for the flight search.

Action: get_location_from_ip

Action Input: None
got data
Country: USThought: I now have the user's current location, which is in the US. Next, I need to find the cheapest flights from the US to Taiwan in the next 24 hours.

Action: tavily_search_results_json

Action Input: "cheapest flights from US to Taiwan in the next 24 hours"
[{'title': 'Best Taiwan Flight Deals - KAYAK', 'url': 'https://www.kayak.com/flight-routes/United-States-US0/Taiwan-TW0', 'content': '#### The cheapest ticket to Taiwan from the United States found in the last 72 hours was **to Taipei City Taiwan Taoyuan Intl Airport, at $692 round-trip**. The most popular route is [New York John F Kennedy Intl Airport (JFK) to Taipei City Taiwan Taoyuan Intl Airport (TPE)](/flight-routes/New-York-John-F-Kennedy-Intl-JFK/Taipei-Taiwan-Taoyuan-Intl-other-Airports-TPE) and the cheapest round-trip ai

{'input': 'Give me the cheapest flight from my current location to Taiwan in the next 24 hrs',
 'output': 'The cheapest flight from the US to Taiwan in the next 24 hours is $323. However, this may not be available on the date you want, and other prices may be around $692 round-trip to Taipei City Taiwan Taoyuan Intl Airport, with the cheapest month to fly being 4 weeks before departure.'}